# **TP3** 
# UC06 - Transformations

In [1]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence_Traitement")
    .master("local[*]")
    .getOrCreate()
)
print("Version Spark : ", spark.version)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 09:54:08 WARN Utils: Your hostname, pauline-XMG-APEX-L23, resolves to a loopback address: 127.0.1.1; using 192.168.1.143 instead (on interface wlp4s0)
26/11/19 09:54:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 09:54:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 09:54:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Version Spark :  4.3.0


## Chargement des données Silver

In [3]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence_Transformation")
    .master("local[*]")
    .getOrCreate()
)

base = os.path.expanduser("~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data")

sales = spark.read.parquet("file://" + base + "/silver/store_sales")
items = spark.read.parquet("file://" + base + "/silver/item")

print("store_sales silver :", sales.count(), "lignes | item silver :", items.count(), "lignes")

26/11/19 10:00:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


store_sales silver : 667579 lignes | item silver : 17820 lignes


## Colonnes calculées

In [4]:
sales = (sales
         .withColumnRenamed("_c15", "montant_vente")
         .withColumn("annee", F.year("date_vente"))
         .withColumn("mois", F.month("date_vente")))

sales.select("date_vente", "annee", "mois", "quantite", "prix_vente", "montant_vente").show(5)

+----------+-----+----+--------+----------+-------------+
|date_vente|annee|mois|quantite|prix_vente|montant_vente|
+----------+-----+----+--------+----------+-------------+
|2001-01-01| 2001|   1|       1|      5.84|         5.84|
|2001-01-01| 2001|   1|       3|     28.03|        84.09|
|2001-01-03| 2001|   1|      51|     47.51|      2423.01|
|2001-01-04| 2001|   1|      30|     23.37|        701.1|
|2001-01-05| 2001|   1|      59|      33.0|       1947.0|
+----------+-----+----+--------+----------+-------------+
only showing top 5 rows


## Jointure ventes et produits + contrôles

In [5]:
produits = items.select("id_produit", "categorie")
ventes_enrichies = sales.join(produits, on="id_produit", how="left")

print("Lignes avant :", sales.count())
print("Lignes après :", ventes_enrichies.count())
print("Ventes sans catégorie :", ventes_enrichies.filter(F.col("categorie").isNull()).count())
ventes_enrichies.select("id_produit", "categorie", "quantite", "montant_vente").show(5)

Lignes avant : 667579
Lignes après : 667579
Ventes sans catégorie : 0
+----------+-----------+--------+-------------+
|id_produit|  categorie|quantite|montant_vente|
+----------+-----------+--------+-------------+
|     10003|Electronics|       1|         5.84|
|     13540|      Music|       3|        84.09|
|     14777|      Books|      51|      2423.01|
|     14229|      Music|      30|        701.1|
|     17422|Movies & TV|      59|       1947.0|
+----------+-----------+--------+-------------+
only showing top 5 rows


## Agrégations

In [6]:
# Granularité : une ligne par catégorie
ca_par_categorie = (ventes_enrichies
    .groupBy("categorie")
    .agg(F.sum("montant_vente").alias("total_ventes"),
         F.sum("quantite").alias("quantite_totale"),
         F.count("*").alias("nombre_lignes"))
    .orderBy(F.desc("total_ventes")))
ca_par_categorie.show(truncate=False)

# Granularité : une ligne par produit
ca_par_produit = (ventes_enrichies
    .groupBy("id_produit", "categorie")
    .agg(F.sum("montant_vente").alias("total_ventes"),
         F.sum("quantite").alias("quantite_totale"))
    .orderBy(F.desc("total_ventes")))
ca_par_produit.show(10, truncate=False)

# Granularité : une ligne par mois
ca_par_mois = (ventes_enrichies
    .groupBy("annee", "mois")
    .agg(F.sum("montant_vente").alias("total_ventes"),
         F.sum("quantite").alias("quantite_totale"))
    .orderBy("annee", "mois"))
ca_par_mois.show(12)

+------------------------+--------------------+---------------+-------------+
|categorie               |total_ventes        |quantite_totale|nombre_lignes|
+------------------------+--------------------+---------------+-------------+
|Music                   |5.773497694399999E8 |15238374       |301236       |
|Books                   |3.405284346899998E8 |9007511        |177671       |
|Movies & TV             |8.302269844000001E7 |2189279        |43376        |
|Clothing & Accessories  |6.2856359919999994E7|1640550        |32372        |
|Electronics             |6.166891152999999E7 |1632098        |32619        |
|Home & Kitchen          |4.589614771000002E7 |1226839        |24380        |
|Sports & Outdoors       |4.561070051999998E7 |1209305        |23911        |
|Toys & Games            |3.0467445000000004E7|809999         |16121        |
|Tools & Home Improvement|2.982353736000001E7 |801519         |15893        |
+------------------------+--------------------+---------------+-

In [9]:
ca_par_annee = (ventes_enrichies.groupBy("annee")
                .agg(F.sum("montant_vente").alias("total_ventes"))
                .orderBy("annee"))
ca_par_annee.show()

+-----+--------------------+
|annee|        total_ventes|
+-----+--------------------+
| 2001|2.5815939533000004E8|
| 2002| 2.569776715199999E8|
| 2003| 2.590600547599999E8|
| 2004|2.6002965673000008E8|
| 2005|2.4299722627000007E8|
+-----+--------------------+



## Vérifier la cohérence

In [7]:
total = ventes_enrichies.agg(F.sum("montant_vente")).first()[0]
total_cat = ca_par_categorie.agg(F.sum("total_ventes")).first()[0]
total_mois = ca_par_mois.agg(F.sum("total_ventes")).first()[0]
nb_lignes = ca_par_categorie.agg(F.sum("nombre_lignes")).first()[0]

print("CA total               :", round(total, 2))
print("Somme par catégorie    :", round(total_cat, 2), "OK" if abs(total - total_cat) < 0.01 else "PROBLÈME")
print("Somme par mois         :", round(total_mois, 2), "OK" if abs(total - total_mois) < 0.01 else "PROBLÈME")
print("Lignes agrégées        :", nb_lignes, "OK" if nb_lignes == ventes_enrichies.count() else "PROBLÈME")

CA total               : 1277224004.61
Somme par catégorie    : 1277224004.61 OK
Somme par mois         : 1277224004.61 OK
Lignes agrégées        : 667579 OK


## Ecrire en Gold

In [10]:
def ecrire(df, nom):
    df.write.mode("overwrite").parquet("file://" + base + "/" + nom)

ecrire(ca_par_categorie, "gold/ca_par_categorie")
ecrire(ca_par_produit, "gold/ca_par_produit")
ecrire(ca_par_mois, "gold/ca_par_mois")
ecrire(ca_par_annee, "gold/ca_par_annee")
print("Écriture terminée")

Écriture terminée


In [11]:
spark.read.parquet("file://" + base + "/gold/ca_par_annee").show()

+-----+--------------------+
|annee|        total_ventes|
+-----+--------------------+
| 2001|2.5815939533000004E8|
| 2002| 2.569776715199999E8|
| 2003| 2.590600547599999E8|
| 2004|2.6002965673000008E8|
| 2005|2.4299722627000007E8|
+-----+--------------------+

